# Matrix Multiplication 5 Ways: Pure Python vs NumPy vs BLAS vs JAX

Same 1000×1000 float64 matrix multiplication (2·10⁹ floating-point operations), five implementations, timed on this machine. Everything below is measured, nothing is quoted from a reference table.

| # | Method | What actually happens |
|---|---|---|
| 1 | Pure Python nested loops | Interpreter executes ~10⁹ inner-loop iterations, each boxing/unboxing Python floats |
| 2 | NumPy row-by-row | 1000 Python-level iterations, each calling a compiled vector-matrix product |
| 3 | NumPy vectorised `A @ B` | One call, dispatches to BLAS `dgemm` |
| 4 | BLAS `dgemm` called directly | Same routine as #3 without NumPy's dispatch layer |
| 5 | JAX `jit` | XLA-compiled; compile once, reuse |


## Environment

In [1]:
import os, time, platform
import numpy as np
from scipy.linalg import blas
from threadpoolctl import threadpool_info, threadpool_limits

N = 1000
PURE_RUNS = 1      # pure Python at n=1000 is ~1e9 inner-loop iterations; one run is enough
FAST_RUNS = 10

print("Python :", platform.python_version())
print("NumPy  :", np.__version__)
print("CPU cores visible:", os.cpu_count())
for lib in threadpool_info():
    print(f"BLAS   : {lib['internal_api']} {lib['version']} | threads={lib['num_threads']} | arch={lib.get('architecture')}")


Python : 3.12.3
NumPy  : 2.4.4
CPU cores visible: 1
BLAS   : openblas 0.3.31.188.0 | threads=1 | arch=SkylakeX
BLAS   : openblas 0.3.30 | threads=1 | arch=SkylakeX


## Implementations

In [2]:
def matmul_pure(A, B, n):
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        Ai, Ci = A[i], C[i]
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += Ai[k] * B[k][j]
            Ci[j] = s
    return C


def matmul_np_rows(A, B):
    n = A.shape[0]
    C = np.empty((n, B.shape[1]))
    for i in range(n):
        C[i] = A[i] @ B
    return C


def matmul_np_vec(A, B):
    return A @ B


def matmul_blas_direct(A, B):
    # C = A @ B  <=>  C^T = B^T @ A^T ; a C-ordered matrix is the F-ordered transpose,
    # so dgemm gets zero-copy inputs.
    return blas.dgemm(1.0, B.T, A.T).T


def timed(fn, runs):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return np.array(times) * 1000  # ms


Correctness first: a fast wrong answer is worthless, so every implementation is checked against `A @ B` on a small size.

In [3]:
rng = np.random.default_rng(0)
A = rng.random((N, N))
B = rng.random((N, N))

n_small = 60
a, b = A[:n_small, :n_small].copy(), B[:n_small, :n_small].copy()
ref = a @ b
assert np.allclose(np.array(matmul_pure(a.tolist(), b.tolist(), n_small)), ref)
assert np.allclose(matmul_np_rows(a, b), ref)
assert np.allclose(matmul_blas_direct(a, b), ref)
print("Correctness: all implementations agree with A @ B (n=60)")


Correctness: all implementations agree with A @ B (n=60)


## Benchmark (n = 1000)

Pure Python is timed once (it takes over a minute); every other method is a 10-run average after a warmup call.

In [4]:
results = {}

Al, Bl = A.tolist(), B.tolist()
results["Pure Python nested loops"] = timed(lambda: matmul_pure(Al, Bl, N), PURE_RUNS)
results["NumPy row-by-row (A[i] @ B)"] = timed(lambda: matmul_np_rows(A, B), FAST_RUNS)

matmul_np_vec(A, B)  # warm caches
results["NumPy vectorised (A @ B)"] = timed(lambda: matmul_np_vec(A, B), FAST_RUNS)

matmul_blas_direct(A, B)
results["BLAS dgemm called directly"] = timed(lambda: matmul_blas_direct(A, B), FAST_RUNS)

import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
f = jax.jit(lambda x, y: x @ y)
Aj, Bj = jnp.asarray(A), jnp.asarray(B)
t0 = time.perf_counter(); f(Aj, Bj).block_until_ready()
compile_ms = (time.perf_counter() - t0) * 1000
results["JAX jit (after warmup)"] = timed(lambda: f(Aj, Bj).block_until_ready(), FAST_RUNS)
print(f"JAX first call (trace + compile + run): {compile_ms:.1f} ms")


JAX first call (trace + compile + run): 41.4 ms


## Results

In [5]:
print(f"{'Method':<32}{'runs':>5}{'mean (ms)':>14}{'std (ms)':>12}")
print("-" * 63)
for name, t in results.items():
    print(f"{name:<32}{len(t):>5}{t.mean():>14.2f}{t.std():>12.2f}")

slowest = max(v.mean() for v in results.values())
fastest = min(v.mean() for v in results.values())
print(f"\nFastest vs slowest: {slowest / fastest:,.0f}x")
pure = results["Pure Python nested loops"].mean()
for name, t in results.items():
    print(f"  {name:<32} {pure / t.mean():>12,.1f}x vs pure Python")

flops = 2 * N ** 3
best = results["NumPy vectorised (A @ B)"].mean() / 1000
print(f"\nA @ B sustained throughput: {flops / best / 1e9:.1f} GFLOPS")


Method                           runs     mean (ms)    std (ms)
---------------------------------------------------------------
Pure Python nested loops            1     155820.98        0.00
NumPy row-by-row (A[i] @ B)        10        257.05       21.28
NumPy vectorised (A @ B)           10         25.88        2.05
BLAS dgemm called directly         10         25.45        2.10
JAX jit (after warmup)             10         27.54        1.04

Fastest vs slowest: 6,122x
  Pure Python nested loops                  1.0x vs pure Python
  NumPy row-by-row (A[i] @ B)             606.2x vs pure Python
  NumPy vectorised (A @ B)              6,021.2x vs pure Python
  BLAS dgemm called directly            6,122.3x vs pure Python
  JAX jit (after warmup)                5,658.4x vs pure Python

A @ B sustained throughput: 77.3 GFLOPS


## Why the gaps are this large

- **Pure Python → NumPy row-by-row:** removes the innermost loop. 10⁶ interpreted iterations become 10³ interpreted iterations plus compiled inner loops.
- **Row-by-row → `A @ B`:** removes the last Python loop. BLAS `dgemm` also tiles the matrices into blocks that fit in CPU cache and uses SIMD instructions, so it reuses each loaded value many times instead of streaming the whole matrix `B` from memory for every row of `A`.
- **`A @ B` vs direct `dgemm`:** the difference is within measurement noise. NumPy's dispatch overhead is negligible at this size.
- **JAX `jit`:** no meaningful gain for a *single* matmul, because both call an optimised matmul kernel. JAX pays off when `jit` can fuse many operations together and avoid materialising intermediate arrays, not for one BLAS-bound op.


## BLAS thread scaling

On a multi-core machine this shows how much of `A @ B` speed comes from parallel BLAS threads. On a single-core machine it prints one line.

In [6]:
print("BLAS thread scaling for A @ B:")
for th in sorted({1, 2, 4, 8, os.cpu_count()}):
    if th > os.cpu_count():
        continue
    with threadpool_limits(limits=th, user_api="blas"):
        t = timed(lambda: matmul_np_vec(A, B), FAST_RUNS)
    print(f"  threads={th}: {t.mean():.2f} ms")


BLAS thread scaling for A @ B:


  threads=1: 25.34 ms


## Memory layout matters

NumPy arrays are row-major (C-contiguous) by default. Reading along a row touches consecutive addresses; reading down a column jumps a full row-length each step. Transposing (`.T`) does not move data, it flips the strides, so the transposed array is no longer C-contiguous.

In [7]:
# Memory layout: same data, same sum, different access pattern.
X = np.random.default_rng(1).random((4000, 4000))   # C-contiguous (row-major)
print("C-contiguous:", X.flags["C_CONTIGUOUS"], "| X.T is C-contiguous:", X.T.flags["C_CONTIGUOUS"])

def sum_rows(M):    # walks contiguous memory
    s = 0.0
    for i in range(M.shape[0]):
        s += M[i, :].sum()
    return s

def sum_cols(M):    # strided access, jumps 4000*8 bytes per element
    s = 0.0
    for j in range(M.shape[1]):
        s += M[:, j].sum()
    return s

t_rows = timed(lambda: sum_rows(X), FAST_RUNS)
t_cols = timed(lambda: sum_cols(X), FAST_RUNS)
print(f"row-wise sums    (contiguous): {t_rows.mean():8.2f} ms")
print(f"column-wise sums (strided)   : {t_cols.mean():8.2f} ms")
print(f"strided is {t_cols.mean() / t_rows.mean():.1f}x slower")


C-contiguous: True | X.T is C-contiguous: False


row-wise sums    (contiguous):    19.36 ms
column-wise sums (strided)   :    33.51 ms
strided is 1.7x slower
